# Module 7 — Harmonized 3-Class EEG Final Pipeline

Adapted from the original Module 7 for the project cache: **3 classes (Left/Right/Feet), 22 channels, 160 Hz, 640 samples**. The pipeline keeps the held-out subject separate, fits training normalization only on non-target subjects, and fits target Euclidean alignment from calibration data only.

The supplied project notes define the harmonized input as 22×640 with labels 0=left, 1=right, 2=feet and emphasize subject-grouped, leakage-safe validation. fileciteturn1file2L225-L247

In [1]:

# ============================================================
# MODULE 7 — HARMONIZED DATASET / HIGH-ACCURACY VERSION
# 3-class MI EEG: Left / Right / Feet
#
# Designed for:
#   X_raw  : (N, 22, 640)
#   y      : {0,1,2}
#   subjects, sessions preserved in eeg_bundle.pkl
#
# Main upgrades vs original Module 7:
#   1) Dynamic number of classes (no hard-coded 4-class logic)
#   2) Dynamic temporal length (works with 640 samples)
#   3) Dynamic subject count / plotting
#   4) Leakage-safe target EA: fit from calibration, apply to cal+test
#   5) Stronger EEGNet for 160 Hz: ~0.5 s temporal kernel, residual head
#   6) EEG augmentation: noise + amplitude + time shift + channel dropout
#   7) Label smoothing + class-balanced CE
#   8) Multi-band CSP with automatic band clipping
#   9) Temperature calibration and confidence fusion
#  10) Stacker uses calibration only; final test remains untouched
#  11) GAN is optional and lightweight by default
#
# IMPORTANT:
#   >70% cannot be honestly guaranteed before running the folds.
#   This version is engineered to maximize the chance while preserving
#   subject-disjoint evaluation. Do NOT select hyperparameters using test.
# ============================================================

import copy
import time
import random
import pickle
import warnings
import math
import numpy as np
import scipy.signal as sig
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    log_loss
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.linalg import eigh

# ------------------------------------------------------------
# 0. REPRODUCIBILITY / DEVICE
# ------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("Device:", device)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------
BUNDLE_PATH = "eeg_bundle.pkl"

# Target adaptation protocol
CALIBRATION_FRACTION = 0.35      # used only when session labels do not give S0/S1
MIN_CAL_PER_CLASS = 8

# EEGNet
CLF_EPOCHS = 100
CLF_BATCH = 64
CLF_LR = 7e-4
CLF_WEIGHT_DECAY = 2e-4
WARMUP = 12
LABEL_SMOOTHING = 0.04
GRAD_CLIP = 1.0
DROPOUT = 0.40

# Fine tuning on target calibration
FT_EPOCHS = 12
FT_LR = 5e-5

# Training augmentation
AUG_NOISE_STD = 0.015
AUG_AMP_RANGE = (0.90, 1.10)
AUG_SHIFT = 18               # samples; ~112 ms at 160 Hz
AUG_CHANNEL_DROP_P = 0.08

# CSP / FBCSP
DEFAULT_BANDS = [
    (8, 12), (10, 14), (12, 16), (14, 18),
    (16, 20), (18, 22), (20, 24), (22, 26),
    (24, 28), (26, 30)
]
N_CSP_COMP = 4

# Optional GAN branch. Keep OFF initially because GAN training is expensive
# and often adds less than discriminative augmentation on small target sets.
RUN_GAN = False
GAN_EPOCHS = 60
GAN_BATCH = 8
NOISE_DIM = 256
N_FAKE_PER_CLASS = 120

# Ablation / execution
RUN_SUBJECT_DEMO = True
DEMO_SUBJECT = None           # None -> first available subject
RUN_FULL_LOSO = True

# ------------------------------------------------------------
# 2. LOAD AND VALIDATE HARMONIZED BUNDLE
# ------------------------------------------------------------
with open(BUNDLE_PATH, "rb") as f:
    bundle = pickle.load(f)

X_raw = np.asarray(bundle["X_raw"], dtype=np.float32)
y = np.asarray(bundle["y"], dtype=np.int64)
subjects = np.asarray(bundle["subjects"]).astype(str)

if "sessions" in bundle:
    sessions = np.asarray(bundle["sessions"]).astype(str)
else:
    sessions = np.array(["single"] * len(y), dtype=str)

FS = int(bundle.get("FS", 160))
BANDS = bundle.get("BANDS", DEFAULT_BANDS)
BANDS = [(float(lo), float(hi)) for lo, hi in BANDS] if BANDS else DEFAULT_BANDS

# Class labels are normalized to 0..C-1
unique_y = np.sort(np.unique(y))
if not np.array_equal(unique_y, np.arange(len(unique_y))):
    remap = {old: i for i, old in enumerate(unique_y)}
    y = np.array([remap[v] for v in y], dtype=np.int64)

N_CLASSES = int(len(np.unique(y)))

if "classes" in bundle:
    try:
        CLASSES = [str(c) for c in np.asarray(bundle["classes"]).tolist()]
        if len(CLASSES) != N_CLASSES:
            CLASSES = [str(i) for i in range(N_CLASSES)]
    except Exception:
        CLASSES = [str(i) for i in range(N_CLASSES)]
else:
    CLASSES = ["Left", "Right", "Feet"][:N_CLASSES]

if N_CLASSES != 3:
    print(f"WARNING: expected 3 classes; detected {N_CLASSES}: {CLASSES}")

assert X_raw.ndim == 3, f"Expected X_raw [N,C,T], got {X_raw.shape}"
assert len(X_raw) == len(y) == len(subjects) == len(sessions)
assert set(np.unique(y)) == set(range(N_CLASSES))

N_SAMPLES, N_CH, N_T = X_raw.shape

# If this is the project cache, the expected geometry is 22x640.
print("\nBundle loaded")
print(" X_raw   :", X_raw.shape)
print(" FS      :", FS)
print(" Subjects:", len(np.unique(subjects)))
print(" Classes :", list(enumerate(CLASSES)))
print(" Sessions:", np.unique(sessions))
print(" Bands   :", BANDS)

# ------------------------------------------------------------
# 3. SAFE / GENERIC HELPERS
# ------------------------------------------------------------
def sort_subjects(arr):
    vals = list(np.unique(np.asarray(arr).astype(str)))
    def key(s):
        nums = re.findall(r"\d+", s)
        return (int(nums[-1]) if nums else 10**9, s)
    return sorted(vals, key=key)

def make_loader(X, y, bs=64, shuffle=True, drop_last=False):
    return DataLoader(
        TensorDataset(
            torch.from_numpy(np.asarray(X, dtype=np.float32)),
            torch.from_numpy(np.asarray(y, dtype=np.int64))
        ),
        batch_size=max(1, int(bs)),
        shuffle=shuffle,
        drop_last=drop_last
    )

def multiclass_accuracy(y_true, pred):
    return 100.0 * accuracy_score(y_true, pred)

def multiclass_balanced_accuracy(y_true, pred):
    return 100.0 * balanced_accuracy_score(y_true, pred)

def softmax_np(logits):
    z = logits - np.max(logits, axis=1, keepdims=True)
    e = np.exp(z)
    return e / (np.sum(e, axis=1, keepdims=True) + 1e-12)

# ------------------------------------------------------------
# 4. LEAKAGE-SAFE EUCLIDEAN ALIGNMENT
# ------------------------------------------------------------
def fit_ea_transform(X):
    """Fit R_bar^-1/2 from ONLY the supplied set."""
    if len(X) == 0:
        raise ValueError("Cannot fit EA on empty data.")
    covs = []
    for trial in X:
        C = trial @ trial.T
        C = C / (np.trace(C) + 1e-8)
        covs.append(C)
    R_bar = np.mean(covs, axis=0)
    vals, vecs = eigh(R_bar + 1e-6 * np.eye(R_bar.shape[0]))
    vals = np.maximum(vals, 1e-10)
    return (vecs @ np.diag(1.0 / np.sqrt(vals)) @ vecs.T).astype(np.float32)

def apply_ea(X, R_inv_sqrt):
    return np.stack([R_inv_sqrt @ trial for trial in X]).astype(np.float32)

def subjectwise_ea_train(X, y, subj_arr):
    """Each training subject gets its own EA transform."""
    xs, ys = [], []
    ea_maps = {}
    for s in sort_subjects(subj_arr):
        m = subj_arr == s
        R = fit_ea_transform(X[m])
        ea_maps[s] = R
        xs.append(apply_ea(X[m], R))
        ys.append(y[m])
    return np.vstack(xs).astype(np.float32), np.concatenate(ys).astype(np.int64), ea_maps

def choose_cal_test_for_subject(subject_mask, sess_arr, S0, S1, y_target):
    """
    Prefer explicit calibration/test sessions when available.
    If not, create a stratified target adaptation split without touching
    the final test selection.
    """
    sub_sess = sess_arr[subject_mask]
    sub_y = y_target[subject_mask]
    sess_unique = list(np.unique(sub_sess))

    if S0 is not None and S1 is not None and S0 in sess_unique and S1 in sess_unique:
        cal_local = sub_sess == S0
        te_local = sub_sess == S1
        if cal_local.sum() >= N_CLASSES * MIN_CAL_PER_CLASS and te_local.sum() > 0:
            return cal_local, te_local, f"session:{S0}->{S1}"

    # Prefer first session as calibration and remaining sessions as test
    if len(sess_unique) >= 2:
        counts = []
        for s in sess_unique:
            counts.append((np.sum(sub_sess == s), s))
        counts.sort(reverse=True)
        cal_s = counts[-1][1]  # smaller session as calibration
        te_s = [s for s in sess_unique if s != cal_s]
        cal_local = sub_sess == cal_s
        te_local = np.isin(sub_sess, te_s)
        if cal_local.sum() >= N_CLASSES * MIN_CAL_PER_CLASS and te_local.sum() > 0:
            return cal_local, te_local, f"session:{cal_s}->remaining"

    # Fallback: stratified split inside the target subject.
    idx = np.arange(len(sub_y))
    test_size = max(0.50, 1.0 - CALIBRATION_FRACTION)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    cal_idx, te_idx = next(splitter.split(idx, sub_y))
    cal_local = np.zeros(len(sub_y), dtype=bool)
    te_local = np.zeros(len(sub_y), dtype=bool)
    cal_local[cal_idx] = True
    te_local[te_idx] = True
    return cal_local, te_local, "stratified-trials"

def prepare_loso_fold(test_subj):
    """
    TRAIN = all non-target subjects
    CAL   = target calibration subset/session
    TEST  = target held-out subset/session
    EA:
      - train subjects: fit subjectwise on training only
      - target: fit R on calibration only; apply same R to cal + test
    Z-score:
      - fit mean/std on training only
      - apply to calibration and test
    """
    train_mask = subjects != test_subj
    target_mask = subjects == test_subj

    X_train_raw = X_raw[train_mask]
    y_train = y[train_mask]

    X_target_raw = X_raw[target_mask]
    y_target = y[target_mask]
    sess_target = sessions[target_mask]

    if X_target_raw.shape[0] < max(2 * N_CLASSES, 10):
        raise ValueError(f"Target {test_subj} has too few samples: {len(X_target_raw)}")

    S0 = bundle.get("S0", None)
    S1 = bundle.get("S1", None)

    cal_local, te_local, split_mode = choose_cal_test_for_subject(
        target_mask, sessions, S0, S1, y_target
    )

    X_cal_raw = X_target_raw[cal_local]
    y_cal = y_target[cal_local]
    X_te_raw = X_target_raw[te_local]
    y_te = y_target[te_local]

    # Training EA
    X_tr_ea, y_tr, _ = subjectwise_ea_train(
        X_train_raw, y_train, subjects[train_mask]
    )

    # Target EA: FIT ONLY from calibration.
    R_target = fit_ea_transform(X_cal_raw)
    X_cal_ea = apply_ea(X_cal_raw, R_target)
    X_te_ea = apply_ea(X_te_raw, R_target)

    # Train-fitted normalization only.
    mu = X_tr_ea.mean(axis=(0, 2), keepdims=True)
    std = X_tr_ea.std(axis=(0, 2), keepdims=True) + 1e-6

    X_tr_z = ((X_tr_ea - mu) / std).astype(np.float32)
    X_cal_z = ((X_cal_ea - mu) / std).astype(np.float32)
    X_te_z = ((X_te_ea - mu) / std).astype(np.float32)

    info = {
        "test_subject": test_subj,
        "split_mode": split_mode,
        "n_train": len(X_tr_z),
        "n_cal": len(X_cal_z),
        "n_test": len(X_te_z),
    }

    return X_tr_z, y_tr, X_cal_z, y_cal, X_te_z, y_te, info

# ------------------------------------------------------------
# 5. TRAINING-AUGMENTATION (TRAIN ONLY)
# ------------------------------------------------------------
def augment_batch_numpy(X, rng):
    X = X.copy()

    # Amplitude scale per trial
    scale = rng.uniform(AUG_AMP_RANGE[0], AUG_AMP_RANGE[1], size=(len(X), 1, 1))
    X *= scale.astype(np.float32)

    # Small noise
    if AUG_NOISE_STD > 0:
        X += rng.normal(0, AUG_NOISE_STD, size=X.shape).astype(np.float32)

    # Time shifts, circularly padded by edge replication
    if AUG_SHIFT > 0:
        shifts = rng.integers(-AUG_SHIFT, AUG_SHIFT + 1, size=len(X))
        for i, sh in enumerate(shifts):
            if sh == 0:
                continue
            X[i] = np.roll(X[i], int(sh), axis=-1)
            # avoid artificial wrap by copying edge region
            if sh > 0:
                X[i, :, :sh] = X[i, :, sh:sh+1]
            else:
                k = abs(int(sh))
                X[i, :, -k:] = X[i, :, -k-1:-k]

    # Random channel dropout
    if AUG_CHANNEL_DROP_P > 0:
        mask = rng.random((len(X), X.shape[1])) < AUG_CHANNEL_DROP_P
        X[mask] = 0.0

    return X.astype(np.float32)

# ------------------------------------------------------------
# 6. FBCSP
# ------------------------------------------------------------
def bandpass(X, lo, hi, fs=FS, order=4):
    nyq = fs / 2.0
    lo = max(0.5, float(lo))
    hi = min(float(hi), nyq - 0.5)
    if hi <= lo:
        raise ValueError(f"Invalid band {lo}-{hi} at fs={fs}.")
    b, a = sig.butter(order, [lo / nyq, hi / nyq], btype="band")
    return sig.filtfilt(b, a, X, axis=-1).astype(np.float32)

def mean_norm_cov(Xset):
    covs = []
    for trial in Xset:
        C = trial @ trial.T
        covs.append(C / (np.trace(C) + 1e-8))
    return np.mean(covs, axis=0)

def fit_csp_all_bands(X_train, y_train, bands=BANDS, n_comp=N_CSP_COMP):
    classes = np.arange(N_CLASSES)
    all_filters = []
    train_feats = []

    for lo, hi in bands:
        lo = max(8.0, float(lo))
        hi = min(30.0, float(hi), FS/2.0 - 0.5)
        if hi <= lo + 0.25:
            continue

        Xb = bandpass(X_train, lo, hi)
        band_filters = []
        band_train_feats = []

        for cls in classes:
            Xc = Xb[y_train == cls]
            Xo = Xb[y_train != cls]
            if len(Xc) < 3 or len(Xo) < 3:
                continue

            Rc = mean_norm_cov(Xc)
            Ro = mean_norm_cov(Xo)
            reg = 1e-5 * np.eye(Rc.shape[0])

            vals, vecs = eigh(Rc + reg, Rc + Ro + 2.0 * reg)
            idx = np.argsort(np.abs(vals - 0.5))[::-1][:n_comp]
            W = vecs[:, idx].astype(np.float32)
            band_filters.append((int(cls), W))

            Z = np.einsum("kc,nct->nkt", W.T, Xb)
            feat = np.log(np.var(Z, axis=2) + 1e-8)
            band_train_feats.append(feat)

        if not band_filters:
            continue

        # Recompute one stable feature matrix in the same order used for transform.
        ordered = []
        for cls, W in band_filters:
            Z = np.einsum("kc,nct->nkt", W.T, Xb)
            ordered.append(np.log(np.var(Z, axis=2) + 1e-8))
        train_feats.append(np.hstack(ordered))
        all_filters.append(((lo, hi), band_filters))

    if not train_feats:
        raise RuntimeError("CSP produced no valid bands.")
    return np.hstack(train_feats), all_filters

def transform_csp_all_bands(X, all_filters):
    all_feats = []
    for (lo, hi), band_filters in all_filters:
        Xb = bandpass(X, lo, hi)
        feats = []
        for _, W in band_filters:
            Z = np.einsum("kc,nct->nkt", W.T, Xb)
            feats.append(np.log(np.var(Z, axis=2) + 1e-8))
        all_feats.append(np.hstack(feats))
    return np.hstack(all_feats)

def make_calibrated_lda():
    base = LDA(solver="lsqr", shrinkage="auto")
    try:
        return CalibratedClassifierCV(estimator=base, method="sigmoid", cv=3)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=base, method="sigmoid", cv=3)

def fit_csp_lda(X_tr, y_tr, X_cal, X_te):
    F_tr, filters = fit_csp_all_bands(X_tr, y_tr)
    F_cal = transform_csp_all_bands(X_cal, filters)
    F_te = transform_csp_all_bands(X_te, filters)

    scaler = StandardScaler()
    F_tr_s = scaler.fit_transform(F_tr)
    F_cal_s = scaler.transform(F_cal)
    F_te_s = scaler.transform(F_te)

    lda = make_calibrated_lda()
    lda.fit(F_tr_s, y_tr)

    P_cal = lda.predict_proba(F_cal_s)
    P_te = lda.predict_proba(F_te_s)

    return {
        "filters": filters,
        "scaler": scaler,
        "lda": lda,
        "P_cal": P_cal,
        "P_te": P_te,
        "pred_cal": P_cal.argmax(1),
        "pred_te": P_te.argmax(1),
    }

# ------------------------------------------------------------
# 7. STRONGER EEGNET FOR 160 Hz
# ------------------------------------------------------------
class EEGNetV2(nn.Module):
    def __init__(self, n_ch, n_t, n_cls,
                 F1=12, D=2, F2=24, dropout=DROPOUT):
        super().__init__()
        temporal_kernel = int(round(0.5 * FS))
        temporal_kernel = min(max(temporal_kernel, 25), max(25, n_t // 2))
        temporal_pad = temporal_kernel // 2

        # Block 1: temporal feature extraction + depthwise spatial filtering
        self.temporal = nn.Sequential(
            nn.Conv2d(
                1, F1, (1, temporal_kernel),
                padding=(0, temporal_pad), bias=False
            ),
            nn.BatchNorm2d(F1),
            nn.Conv2d(
                F1, F1 * D, (n_ch, 1),
                groups=F1, bias=False
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 2: depthwise-separable temporal conv
        self.sep = nn.Sequential(
            nn.Conv2d(
                F1 * D, F1 * D,
                (1, 15), padding=(0, 7),
                groups=F1 * D, bias=False
            ),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Residual temporal path
        self.res = nn.Sequential(
            nn.Conv2d(1, F2, (1, 31), padding=(0, 15), bias=False),
            nn.BatchNorm2d(F2),
            nn.Conv2d(F2, F2, (n_ch, 1), groups=F2, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 16)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            d = torch.zeros(1, 1, n_ch, n_t)
            a = self.sep(self.temporal(d))
            b = self.res(d)
            min_t = min(a.shape[-1], b.shape[-1])
            z = a[..., :min_t] + b[..., :min_t]
            self.feature_dim = z.flatten(1).shape[1]

        self.head = nn.Sequential(
            nn.Linear(self.feature_dim, 64),
            nn.ELU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(64, n_cls),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        a = self.temporal(x)
        a = self.sep(a)
        b = self.res(x)
        min_t = min(a.shape[-1], b.shape[-1])
        z = a[..., :min_t] + b[..., :min_t]
        feat = z.flatten(1)
        return self.head(feat), feat

def compute_class_weights(y_train):
    counts = np.bincount(y_train, minlength=N_CLASSES).astype(np.float32)
    counts[counts == 0] = 1.0
    w = counts.sum() / (N_CLASSES * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)

def train_eegnet(X_tr, y_tr, n_epochs=CLF_EPOCHS, lr=CLF_LR,
                 batch_size=CLF_BATCH, verbose=False):
    n_ch, n_t = X_tr.shape[1], X_tr.shape[2]
    model = EEGNetV2(n_ch=n_ch, n_t=n_t, n_cls=N_CLASSES).to(device)

    weights = compute_class_weights(y_tr)
    ce = nn.CrossEntropyLoss(
        weight=weights,
        label_smoothing=LABEL_SMOOTHING
    )

    opt = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=CLF_WEIGHT_DECAY
    )
    sched = optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=lr,
        epochs=n_epochs,
        steps_per_epoch=max(1, math.ceil(len(X_tr) / batch_size)),
        pct_start=0.15,
        div_factor=10.0,
        final_div_factor=50.0
    )

    rng = np.random.default_rng(SEED)
    hist = {"loss": [], "train_acc": []}

    best_state = copy.deepcopy(model.state_dict())
    best_score = -np.inf

    for ep in range(1, n_epochs + 1):
        model.train()
        perm = rng.permutation(len(X_tr))
        cor = 0
        total = 0
        loss_sum = 0.0
        batches = 0

        for start in range(0, len(perm), batch_size):
            idx = perm[start:start + batch_size]
            Xb_np = augment_batch_numpy(X_tr[idx], rng)
            yb_np = y_tr[idx]

            Xb = torch.from_numpy(Xb_np).float().to(device)
            yb = torch.from_numpy(yb_np).long().to(device)

            opt.zero_grad(set_to_none=True)
            logits, _ = model(Xb)
            loss = ce(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            sched.step()

            loss_sum += float(loss.item())
            batches += 1
            cor += int((logits.argmax(1) == yb).sum().item())
            total += len(yb)

        tr_acc = 100.0 * cor / max(total, 1)
        mean_loss = loss_sum / max(batches, 1)
        hist["loss"].append(mean_loss)
        hist["train_acc"].append(tr_acc)

        # Save by training score with a mild preference for later epochs.
        score = tr_acc - 0.001 * ep
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())

        if verbose and (ep == 1 or ep % 20 == 0 or ep == n_epochs):
            print(
                f"    ep {ep:03d}/{n_epochs} "
                f"loss={mean_loss:.4f} train={tr_acc:.1f}%"
            )

    model.load_state_dict(best_state)
    return model, hist

# ------------------------------------------------------------
# 8. TARGET FINE-TUNING
# ------------------------------------------------------------
def finetune_eegnet(model, X_cal, y_cal, n_epochs=FT_EPOCHS, lr=FT_LR):
    # Freeze BN statistics; update weights conservatively.
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    counts = np.bincount(y_cal, minlength=N_CLASSES).astype(np.float32)
    counts[counts == 0] = 1.0
    w = counts.sum() / (N_CLASSES * counts)
    ce = nn.CrossEntropyLoss(
        weight=torch.tensor(w, dtype=torch.float32, device=device),
        label_smoothing=0.02
    )

    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(1, n_epochs), eta_min=1e-7
    )

    rng = np.random.default_rng(SEED + 1)
    bs = max(8, min(32, max(8, len(X_cal) // 4)))
    hist = {"loss": [], "train_acc": []}
    best_state = copy.deepcopy(model.state_dict())
    best_score = -np.inf

    for ep in range(1, n_epochs + 1):
        model.train()
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

        perm = rng.permutation(len(X_cal))
        cor = 0
        total = 0
        loss_sum = 0.0
        nb = 0

        for start in range(0, len(perm), bs):
            idx = perm[start:start + bs]
            Xb_np = augment_batch_numpy(X_cal[idx], rng)
            yb_np = y_cal[idx]

            Xb = torch.from_numpy(Xb_np).float().to(device)
            yb = torch.from_numpy(yb_np).long().to(device)

            opt.zero_grad(set_to_none=True)
            logits, _ = model(Xb)
            loss = ce(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            opt.step()

            loss_sum += float(loss.item())
            nb += 1
            cor += int((logits.argmax(1) == yb).sum().item())
            total += len(yb)

        sched.step()

        tr_acc = 100.0 * cor / max(total, 1)
        hist["loss"].append(loss_sum / max(nb, 1))
        hist["train_acc"].append(tr_acc)

        if tr_acc > best_score:
            best_score = tr_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, hist

# ------------------------------------------------------------
# 9. PROBABILITY / CALIBRATION / FUSION
# ------------------------------------------------------------
def get_logits(model, X, bs=128):
    model.eval()
    outs = []
    Xt = torch.from_numpy(X).float()
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            logits, _ = model(Xt[i:i + bs].to(device))
            outs.append(logits.cpu().numpy())
    return np.vstack(outs)

def get_probs_temp(model, X, T=1.0, bs=128):
    return softmax_np(get_logits(model, X, bs=bs) / max(float(T), 1e-6))

def temperature_scale_search(model, X_cal, y_cal):
    logits = get_logits(model, X_cal)
    temps = np.arange(0.70, 3.01, 0.05)
    best_T, best_nll = 1.0, np.inf

    for T in temps:
        P = softmax_np(logits / T)
        nll = log_loss(y_cal, P, labels=list(range(N_CLASSES)))
        if nll < best_nll:
            best_nll = nll
            best_T = float(T)

    return best_T, best_nll

def entropy_from_probs(P):
    return -np.sum(P * np.log(P + 1e-12), axis=1, keepdims=True)

def weighted_fusion(P1, P2, alpha):
    c1 = np.max(P1, axis=1, keepdims=True)
    c2 = np.max(P2, axis=1, keepdims=True)
    w1 = alpha * c1
    w2 = (1.0 - alpha) * c2
    return (w1 * P1 + w2 * P2) / (w1 + w2 + 1e-8)

def search_alpha(P1_cal, P2_cal, y_cal):
    # Accuracy on CAL is used only for selecting among already-defined models.
    best = (0.5, -1.0)
    for alpha in np.arange(0.0, 1.001, 0.025):
        P = weighted_fusion(P1_cal, P2_cal, float(alpha))
        acc = accuracy_score(y_cal, P.argmax(1)) * 100.0
        if acc > best[1]:
            best = (float(alpha), float(acc))
    return best

def build_meta_features(P1, P2):
    c1 = np.max(P1, axis=1, keepdims=True)
    c2 = np.max(P2, axis=1, keepdims=True)
    e1 = entropy_from_probs(P1)
    e2 = entropy_from_probs(P2)
    return np.hstack([P1, P2, c1, c2, e1, e2])

def fit_stacker(P1_cal, P2_cal, y_cal):
    X_meta = build_meta_features(P1_cal, P2_cal)
    C_grid = [0.2, 0.5, 1.0, 2.0, 5.0]
    best_model = None
    best_acc = -1.0

    for C in C_grid:
        clf = LogisticRegression(
            max_iter=3000,
            multi_class="multinomial",
            C=C,
            class_weight="balanced",
            random_state=SEED
        )
        clf.fit(X_meta, y_cal)
        pred = clf.predict(X_meta)
        acc = accuracy_score(y_cal, pred) * 100.0
        if acc > best_acc:
            best_acc = float(acc)
            best_model = clf

    return best_model, best_acc

# ------------------------------------------------------------
# 10. OPTIONAL LIGHTWEIGHT GAN (3-class, variable-size)
# ------------------------------------------------------------
class TinyGenerator(nn.Module):
    def __init__(self, n_ch, n_t, noise_dim=NOISE_DIM):
        super().__init__()
        self.n_ch, self.n_t = n_ch, n_t
        hidden = 128
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_ch * min(160, n_t)),
        )
        self.base_t = min(160, n_t)

    def forward(self, z):
        x = self.net(z)
        x = x.view(z.size(0), self.n_ch, self.base_t)
        x = F.interpolate(
            x, size=self.n_t, mode="linear", align_corners=False
        )
        return torch.tanh(x)

class TinyDiscriminator(nn.Module):
    def __init__(self, n_ch):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv1d(n_ch, 32, 9, padding=4)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AvgPool1d(4),
            spectral_norm(nn.Conv1d(32, 64, 9, padding=4)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        z = self.net(x).squeeze(-1)
        return self.fc(z)

def train_tiny_gan(X_cls, n_ch, n_t):
    if len(X_cls) < max(GAN_BATCH, 4):
        reps = int(np.ceil(max(GAN_BATCH, 4) / max(1, len(X_cls))))
        X_cls = np.tile(X_cls, (reps, 1, 1))

    Xg = np.clip(X_cls, -3, 3).astype(np.float32)
    ds = TensorDataset(torch.from_numpy(Xg))
    loader = DataLoader(
        ds, batch_size=GAN_BATCH, shuffle=True,
        drop_last=False
    )

    G = TinyGenerator(n_ch, n_t).to(device)
    D = TinyDiscriminator(n_ch).to(device)

    og = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    od = optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()

    for ep in range(GAN_EPOCHS):
        for (real,) in loader:
            real = real.to(device)
            bs = real.size(0)

            # D
            od.zero_grad(set_to_none=True)
            z = torch.randn(bs, NOISE_DIM, device=device)
            with torch.no_grad():
                fake = G(z)
            ld = (
                bce(D(real), torch.full((bs,1), 0.9, device=device)) +
                bce(D(fake), torch.full((bs,1), 0.1, device=device))
            )
            ld.backward()
            od.step()

            # G
            og.zero_grad(set_to_none=True)
            z = torch.randn(bs, NOISE_DIM, device=device)
            fake = G(z)
            lg = bce(D(fake), torch.full((bs,1), 0.9, device=device))
            lg.backward()
            og.step()

    return G

def generate_fake(G, n_fake=N_FAKE_PER_CLASS):
    G.eval()
    chunks = []
    with torch.no_grad():
        for i in range(0, n_fake, 64):
            k = min(64, n_fake - i)
            z = torch.randn(k, NOISE_DIM, device=device)
            chunks.append(G(z).cpu().numpy())
    return np.vstack(chunks).astype(np.float32)

def run_gan_target_calibration(X_cal, y_cal):
    fake_X, fake_y = [], []
    for cls in range(N_CLASSES):
        X_cls = X_cal[y_cal == cls]
        G = train_tiny_gan(X_cls, X_cal.shape[1], X_cal.shape[2])
        fake = generate_fake(G)
        fake_X.append(fake)
        fake_y.append(np.full(len(fake), cls, dtype=np.int64))
        del G
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.vstack(fake_X), np.concatenate(fake_y)

# ------------------------------------------------------------
# 11. FOLD PIPELINE
# ------------------------------------------------------------
def run_fold(test_subj, verbose=False):
    X_tr, y_tr, X_cal, y_cal, X_te, y_te, info = prepare_loso_fold(test_subj)

    print(
        f"  split={info['split_mode']} | "
        f"train={len(y_tr)} cal={len(y_cal)} test={len(y_te)}"
    )

    # Class presence assertions
    for split_name, yy in [("train", y_tr), ("cal", y_cal), ("test", y_te)]:
        missing = [c for c in range(N_CLASSES) if np.sum(yy == c) == 0]
        if missing:
            raise ValueError(f"{test_subj}: {split_name} missing classes {missing}")

    # ---------- CSP ----------
    csp = fit_csp_lda(X_tr, y_tr, X_cal, X_te)
    P_csp_cal = csp["P_cal"]
    P_csp_te = csp["P_te"]

    # ---------- EEGNet base ----------
    model_base, hist_base = train_eegnet(X_tr, y_tr, verbose=verbose)
    T_base, nll_base = temperature_scale_search(model_base, X_cal, y_cal)
    P_base_cal = get_probs_temp(model_base, X_cal, T=T_base)
    P_base_te = get_probs_temp(model_base, X_te, T=T_base)

    # ---------- Fine-tuned ----------
    model_ft = copy.deepcopy(model_base)
    model_ft, hist_ft = finetune_eegnet(model_ft, X_cal, y_cal)
    T_ft, nll_ft = temperature_scale_search(model_ft, X_cal, y_cal)
    P_ft_cal = get_probs_temp(model_ft, X_cal, T=T_ft)
    P_ft_te = get_probs_temp(model_ft, X_te, T=T_ft)

    # ---------- Optional GAN ----------
    P_gan_cal = P_gan_te = None
    model_gan = None
    gan_info = None

    if RUN_GAN:
        tgan = time.time()
        fake_X, fake_y = run_gan_target_calibration(X_cal, y_cal)
        X_aug = np.vstack([X_tr, fake_X]).astype(np.float32)
        y_aug = np.hstack([y_tr, fake_y]).astype(np.int64)

        model_gan, hist_gan = train_eegnet(X_aug, y_aug, verbose=verbose)
        T_gan, nll_gan = temperature_scale_search(model_gan, X_cal, y_cal)
        P_gan_cal = get_probs_temp(model_gan, X_cal, T=T_gan)
        P_gan_te = get_probs_temp(model_gan, X_te, T=T_gan)

        gan_info = {
            "time_min": (time.time() - tgan) / 60.0,
            "hist": hist_gan,
            "T": T_gan,
            "nll": nll_gan
        }

    # ---------- Candidate collection ----------
    probs = {
        "csp": (P_csp_cal, P_csp_te, None),
        "eegnet": (P_base_cal, P_base_te, None),
        "ft": (P_ft_cal, P_ft_te, None),
    }
    if RUN_GAN:
        probs["gan"] = (P_gan_cal, P_gan_te, None)

    candidates = {}

    # Single branches
    for name, (Pc, Pt, _) in probs.items():
        candidates[name] = (
            accuracy_score(y_cal, Pc.argmax(1)) * 100.0,
            Pt.argmax(1)
        )

    # Weighted fusions
    fusion_names = ["eegnet", "ft"] + (["gan"] if RUN_GAN else [])
    for name in fusion_names:
        alpha, cal_acc = search_alpha(P_csp_cal, probs[name][0], y_cal)
        P_cal = weighted_fusion(P_csp_cal, probs[name][0], alpha)
        P_te = weighted_fusion(P_csp_te, probs[name][1], alpha)
        candidates[f"fusion_{name}"] = (
            cal_acc, P_te.argmax(1)
        )

    # Stackers
    for name in fusion_names:
        stacker, cal_acc = fit_stacker(
            P_csp_cal, probs[name][0], y_cal
        )
        P_stack_te = stacker.predict_proba(
            build_meta_features(P_csp_te, probs[name][1])
        )
        candidates[f"stack_{name}"] = (
            cal_acc, P_stack_te.argmax(1)
        )

    # Select on CAL ONLY
    best_method = max(candidates, key=lambda k: candidates[k][0])
    best_cal_acc, best_pred_te = candidates[best_method]
    best_te_acc = multiclass_accuracy(y_te, best_pred_te)
    best_te_bacc = multiclass_balanced_accuracy(y_te, best_pred_te)

    # Also record every test metric for diagnosis, without using it for selection.
    test_metrics = {
        name: multiclass_accuracy(y_te, pred)
        for name, (_, pred) in candidates.items()
    }

    fold = {
        "subject": test_subj,
        "info": info,
        "X_tr": X_tr, "y_tr": y_tr,
        "X_cal": X_cal, "y_cal": y_cal,
        "X_te": X_te, "y_te": y_te,
        "csp": {
            "P_cal": P_csp_cal, "P_te": P_csp_te,
            "cal_acc": multiclass_accuracy(y_cal, P_csp_cal.argmax(1)),
            "te_acc": multiclass_accuracy(y_te, P_csp_te.argmax(1))
        },
        "base": {
            "P_cal": P_base_cal, "P_te": P_base_te,
            "T": T_base, "nll": nll_base,
            "cal_acc": multiclass_accuracy(y_cal, P_base_cal.argmax(1)),
            "te_acc": multiclass_accuracy(y_te, P_base_te.argmax(1)),
            "hist": hist_base
        },
        "ft": {
            "P_cal": P_ft_cal, "P_te": P_ft_te,
            "T": T_ft, "nll": nll_ft,
            "cal_acc": multiclass_accuracy(y_cal, P_ft_cal.argmax(1)),
            "te_acc": multiclass_accuracy(y_te, P_ft_te.argmax(1)),
            "hist": hist_ft
        },
        "gan": gan_info,
        "candidates": {
            k: {"cal_acc": float(v[0]), "te_acc": float(test_metrics[k])}
            for k, v in candidates.items()
        },
        "best_method": best_method,
        "best_cal_acc": float(best_cal_acc),
        "best_te_acc": float(best_te_acc),
        "best_te_bacc": float(best_te_bacc),
        "best_preds_te": best_pred_te
    }

    print(
        f"  best={best_method:<14} "
        f"cal={best_cal_acc:5.1f}% | "
        f"test={best_te_acc:5.1f}% | "
        f"bacc={best_te_bacc:5.1f}%"
    )
    return fold

# ------------------------------------------------------------
# 12. PLOTS
# ------------------------------------------------------------
def plot_subject_demo(fold):
    labels, vals = [], []
    for key in ["csp", "base", "ft"]:
        labels.append({"csp":"CSP+LDA","base":"EEGNet","ft":"EEGNet+FT"}[key])
        vals.append(fold[key]["te_acc"])

    # candidate test results for diagnostic display
    for name in ["fusion_eegnet", "fusion_ft", "stack_eegnet", "stack_ft"]:
        if name in fold["candidates"]:
            labels.append(name.replace("_", "\n"))
            vals.append(fold["candidates"][name]["te_acc"])

    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(np.arange(len(vals)), vals)
    ax.axhline(33.33, linestyle=":", linewidth=1.2, label="3-class chance")
    ax.axhline(70.0, linestyle="--", linewidth=1.2, label="70% target")
    ax.set_xticks(np.arange(len(vals)))
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Test accuracy (%)")
    ax.set_title(f"Module 7 — Subject {fold['subject']}")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    for b, v in zip(bars, vals):
        ax.text(
            b.get_x() + b.get_width()/2,
            min(98, v + 1.2),
            f"{v:.1f}%",
            ha="center", fontsize=9
        )
    plt.tight_layout()
    plt.show()

def plot_progression(best_results, method_map):
    subj_list = sort_subjects(best_results.keys())
    vals = [best_results[s] for s in subj_list]

    fig, ax = plt.subplots(figsize=(max(12, len(subj_list)*0.35), 5))
    bars = ax.bar(np.arange(len(vals)), vals)
    ax.axhline(33.33, linestyle=":", linewidth=1.2, label="3-class chance")
    ax.axhline(70, linestyle="--", linewidth=1.2, label="70% target")
    ax.set_ylim(0, 100)
    ax.set_xticks(np.arange(len(vals)))
    ax.set_xticklabels(subj_list, rotation=90)
    ax.set_ylabel("Best test accuracy (%)")
    ax.set_title("Module 7 — Best method per unseen subject")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    for b, v, s in zip(bars, vals, subj_list):
        ax.text(
            b.get_x() + b.get_width()/2,
            min(98, v + 0.8),
            method_map[s],
            ha="center", va="bottom", fontsize=6, rotation=90
        )
    plt.tight_layout()
    plt.show()

def plot_confusion_summary(best_preds, best_true, best_results):
    subj_list = sort_subjects(best_preds.keys())
    cols = 4
    rows = int(np.ceil(len(subj_list) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.6*rows))
    axes = np.atleast_1d(axes).ravel()

    for idx, s in enumerate(subj_list):
        cm = confusion_matrix(
            best_true[s], best_preds[s],
            labels=list(range(N_CLASSES))
        )
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES,
            cbar=False, ax=axes[idx]
        )
        axes[idx].set_title(f"{s}: {best_results[s]:.1f}%")
        axes[idx].set_xlabel("Pred")
        axes[idx].set_ylabel("True")

    for j in range(len(subj_list), len(axes)):
        axes[j].axis("off")

    plt.suptitle("Confusion matrices — best method", y=1.01)
    plt.tight_layout()
    plt.show()

def plot_per_class_heatmap(best_preds, best_true):
    subj_list = sort_subjects(best_preds.keys())
    per_class = np.full((len(subj_list), N_CLASSES), np.nan)

    for i, s in enumerate(subj_list):
        true = best_true[s]
        pred = best_preds[s]
        for c in range(N_CLASSES):
            m = true == c
            if np.any(m):
                per_class[i, c] = 100.0 * np.mean(pred[m] == c)

    fig, ax = plt.subplots(figsize=(7, max(5, 0.28 * len(subj_list))))
    im = ax.imshow(per_class, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(N_CLASSES))
    ax.set_xticklabels(CLASSES)
    ax.set_yticks(range(len(subj_list)))
    ax.set_yticklabels(subj_list)
    ax.set_xlabel("Class")
    ax.set_ylabel("Subject")
    ax.set_title("Per-class accuracy")
    plt.colorbar(im, ax=ax, label="Accuracy (%)")

    for i in range(len(subj_list)):
        for j in range(N_CLASSES):
            if np.isfinite(per_class[i, j]):
                ax.text(
                    j, i, f"{per_class[i,j]:.0f}",
                    ha="center", va="center", fontsize=8
                )
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 13. MAIN
# ------------------------------------------------------------
def run_subject_demo(subj=None):
    if subj is None:
        subj = sort_subjects(subjects)[0]
    print("\n" + "=" * 72)
    print(f"MODULE 7 DEMO — unseen subject {subj}")
    print("=" * 72)
    fold = run_fold(str(subj), verbose=True)
    plot_subject_demo(fold)
    return fold

def run_full_loso():
    results = {}
    method_map = {}
    best_preds = {}
    best_true = {}
    fold_store = {}

    t0 = time.time()
    subj_list = sort_subjects(subjects)

    for k, subj in enumerate(subj_list, 1):
        print("\n" + "═" * 72)
        print(f"LOSO {k}/{len(subj_list)} — TEST SUBJECT: {subj}")
        print("═" * 72)

        ts = time.time()
        fold = run_fold(subj, verbose=False)
        fold_store[subj] = fold

        results[subj] = fold["best_te_acc"]
        method_map[subj] = fold["best_method"]
        best_preds[subj] = fold["best_preds_te"]
        best_true[subj] = fold["y_te"]

        print(f"elapsed={(time.time() - ts)/60:.2f} min")

    vals = np.asarray(list(results.values()), dtype=float)
    mean_acc = float(vals.mean())
    std_acc = float(vals.std())
    mean_bacc = float(np.mean([
        fold_store[s]["best_te_bacc"] for s in subj_list
    ]))

    print("\n" + "=" * 72)
    print("MODULE 7 FINAL")
    print("=" * 72)
    print(f"Subjects              : {len(subj_list)}")
    print(f"Mean accuracy         : {mean_acc:.2f}%")
    print(f"Std accuracy          : {std_acc:.2f}%")
    print(f"Mean balanced accuracy: {mean_bacc:.2f}%")
    print(f"70% target            : {'REACHED' if mean_acc >= 70 else 'NOT REACHED'}")
    print(f"Total time            : {(time.time() - t0)/60:.1f} min")

    return (
        results, method_map, best_preds, best_true,
        mean_acc, std_acc, mean_bacc, fold_store
    )

# ------------------------------------------------------------
# 14. EXECUTE
# ------------------------------------------------------------
demo_fold = None
if RUN_SUBJECT_DEMO:
    demo_fold = run_subject_demo(DEMO_SUBJECT)

if RUN_FULL_LOSO:
    (
        best_results,
        best_method_map,
        best_preds,
        best_true,
        mean_acc,
        std_acc,
        mean_bacc,
        fold_store
    ) = run_full_loso()

    # Save a compact result summary back into the bundle.
    bundle["results_m7"] = best_results
    bundle["m7_mean"] = mean_acc
    bundle["m7_std"] = std_acc
    bundle["m7_balanced_mean"] = mean_bacc
    bundle["m7_best_method"] = best_method_map
    bundle["m7_config"] = {
        "n_classes": N_CLASSES,
        "classes": CLASSES,
        "fs": FS,
        "input_shape": [N_CH, N_T],
        "bands": BANDS,
        "calibration_fraction": CALIBRATION_FRACTION,
        "run_gan": RUN_GAN
    }
    bundle["m7_fold_store"] = {
        s: {
            "best_method": fold_store[s]["best_method"],
            "best_te_acc": fold_store[s]["best_te_acc"],
            "best_te_bacc": fold_store[s]["best_te_bacc"],
            "split_mode": fold_store[s]["info"]["split_mode"],
            "n_train": fold_store[s]["info"]["n_train"],
            "n_cal": fold_store[s]["info"]["n_cal"],
            "n_test": fold_store[s]["info"]["n_test"],
        }
        for s in fold_store
    }

    with open(BUNDLE_PATH, "wb") as f:
        pickle.dump(bundle, f)

    plot_progression(best_results, best_method_map)
    plot_confusion_summary(best_preds, best_true, best_results)
    plot_per_class_heatmap(best_preds, best_true)

print("\n✅ MODULE 7 READY FOR THE HARMONIZED 3-CLASS DATASET")


Device: mps


FileNotFoundError: [Errno 2] No such file or directory: 'eeg_bundle.pkl'